Devadharshan S

CH.SC.U4CSE24113

1. Maggie is a real-estate agent and she is puzzled about why some of the properties her company manages have
not been sold for more than six months. Help Maggie in providing the insights why the properties are not getting
sold.

In [ ]:
# Importing Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Creating a Real Estate Dataset
np.random.seed(42)
n = 200

data = pd.DataFrame({
    'price': np.random.normal(350000, 100000, n),
    'area_sqft': np.random.normal(1800, 500, n),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n),
    'bathrooms': np.random.choice([1, 2, 3], n),
    'age_years': np.random.randint(0, 50, n),
    'distance_to_city': np.random.normal(15, 8, n),
    'parking_spaces': np.random.choice([0, 1, 2, 3], n),
    'condition_score': np.random.randint(1, 11, n),
    'neighborhood_score': np.random.randint(1, 11, n)
})

# Label: 1 = sold within 6 months, 0 = not sold
sold = []
for i in range(n):
    score = 0
    if data['price'].iloc[i] < 350000: score += 1
    if data['condition_score'].iloc[i] > 5: score += 1
    if data['neighborhood_score'].iloc[i] > 5: score += 1
    if data['distance_to_city'].iloc[i] < 15: score += 1
    if data['age_years'].iloc[i] < 20: score += 1
    sold.append(1 if score >= 3 else 0)

data['sold'] = sold
print(data.shape)
data.head()

In [ ]:
# Checking for any null values in dataset
data.isnull().sum()

In [ ]:
# Assigned variables
feature_cols = ['price', 'area_sqft', 'bedrooms', 'bathrooms',
                'age_years', 'distance_to_city', 'parking_spaces',
                'condition_score', 'neighborhood_score']
print('Features:', feature_cols)
print('Target: sold (1 = Sold within 6 months, 0 = Not Sold)')

In [ ]:
# Creating dataframe with features only
df1 = data[feature_cols]
df1.head()

In [ ]:
# Standard Scaler and PCA
sc = StandardScaler()
sc.fit(df1)
scaled_data = sc.transform(df1)
principal = PCA(n_components=3)
principal.fit(scaled_data)
x = principal.transform(scaled_data)
print(x.shape)

In [ ]:
# Scattering the Clusters - PCA 2D
principal.components_
plt.figure(figsize=(10, 10))
plt.title('PCA 2D')
plt.scatter(x[:, 0], x[:, 1], c=data['sold'], cmap='plasma')
plt.xlabel('pc1')
plt.ylabel('pc2')
plt.colorbar(label='Sold (1) / Not Sold (0)')
plt.show()

In [ ]:
# Scattering the Clusters - PCA 3D
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10, 10))
axis = fig.add_subplot(111, projection='3d')
axis.scatter(x[:, 0], x[:, 1], x[:, 2], c=data['sold'], cmap='plasma')
axis.set_xlabel('PC1', fontsize=10)
axis.set_ylabel('PC2', fontsize=10)
axis.set_zlabel('PC3', fontsize=10)
plt.title('PCA 3D - Real Estate Properties')
plt.show()

In [ ]:
# Explained Variance Ratio
print('Explained Variance Ratio:', principal.explained_variance_ratio_)
print('\nTotal Variance Explained by 3 components:',
      sum(principal.explained_variance_ratio_))

In [ ]:
# Feature Importance - Which features contribute most to each principal component
pc_df = pd.DataFrame(
    principal.components_,
    columns=feature_cols,
    index=['PC1', 'PC2', 'PC3']
)
print('Principal Component Loadings:')
pc_df

In [ ]:
# Insight: Properties not sold tend to have high price, old age, far from city,
# low condition and neighborhood scores
not_sold = data[data['sold'] == 0]
sold_props = data[data['sold'] == 1]

print('=== Insights for Maggie ===')
print(f'\nTotal Properties: {len(data)}')
print(f'Sold within 6 months: {len(sold_props)}')
print(f'Not Sold: {len(not_sold)}')
print(f'\n--- Average values comparison ---')
comparison = pd.DataFrame({
    'Sold (Avg)': sold_props[feature_cols].mean(),
    'Not Sold (Avg)': not_sold[feature_cols].mean()
})
comparison